In [0]:
# =============================================================
# Notebook : 02_gold_customer_rfm.py
# Purpose  : RFM segmentation — who are our best customers?
# Source   : walmart_silver.fact_pos_transactions
#            walmart_silver.dim_customer
# Target   : gold/gold_customer_rfm/
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

GOLD_PATH = "abfss://gold@walmartdata.dfs.core.windows.net/"

# ── Step 1: Compute RFM metrics per customer ──────────────────
df_rfm_raw = spark.sql("""
    SELECT
        customer_loyalty_id                         AS customer_id,
        DATEDIFF(current_date(),
                 MAX(sale_date))                    AS recency_days,
        COUNT(DISTINCT transaction_id)              AS frequency,
        ROUND(SUM(total_amount), 2)                 AS monetary_value,
        MAX(sale_date)                              AS last_purchase_date,
        MIN(sale_date)                              AS first_purchase_date,
        AVG(total_amount)                           AS avg_order_value,
        COUNT(DISTINCT store_id)                    AS stores_visited
    FROM walmart_silver.fact_pos_transactions
    WHERE customer_loyalty_id IS NOT NULL
    GROUP BY customer_loyalty_id
""")

print(f"Customers with transactions: {df_rfm_raw.count():,}")
display(df_rfm_raw.limit(5))

In [0]:
# ── Step 2: Score R, F, M on 1-5 scale ───────────────────────
# Recency   : lower days = better = score 5
# Frequency : higher = better = score 5
# Monetary  : higher = better = score 5

df_rfm_scored = (
    df_rfm_raw
    .withColumn("r_score",
        F.when(F.col("recency_days") <= 7,  5)
         .when(F.col("recency_days") <= 14, 4)
         .when(F.col("recency_days") <= 30, 3)
         .when(F.col("recency_days") <= 60, 2)
         .otherwise(1))
    .withColumn("f_score",
        F.when(F.col("frequency") >= 10, 5)
         .when(F.col("frequency") >= 7,  4)
         .when(F.col("frequency") >= 4,  3)
         .when(F.col("frequency") >= 2,  2)
         .otherwise(1))
    .withColumn("m_score",
        F.when(F.col("monetary_value") >= 50000, 5)
         .when(F.col("monetary_value") >= 20000, 4)
         .when(F.col("monetary_value") >= 10000, 3)
         .when(F.col("monetary_value") >= 5000,  2)
         .otherwise(1))
    .withColumn("rfm_score",
        F.col("r_score") + F.col("f_score") + F.col("m_score"))
    .withColumn("rfm_segment",
        F.when(F.col("rfm_score") >= 13, "Champions")
         .when(F.col("rfm_score") >= 10, "Loyal Customers")
         .when(F.col("rfm_score") >= 7,  "Potential Loyalists")
         .when(F.col("rfm_score") >= 5,  "At Risk")
         .otherwise("Lost Customers"))
    .withColumn("gold_processed_at", F.current_timestamp())
)

print("RFM Segment distribution:")
df_rfm_scored.groupBy("rfm_segment") \
             .agg(
                 F.count("*").alias("customers"),
                 F.round(F.avg("monetary_value"), 2).alias("avg_spend"),
                 F.round(F.avg("frequency"), 1).alias("avg_orders")
             ) \
             .orderBy("customers", ascending=False) \
             .display()

In [0]:
# ── Step 3: Join with CRM profile ────────────────────────────
df_rfm_final = (
    df_rfm_scored.alias("r")
    .join(
        spark.table("walmart_silver.dim_customer").alias("c"),
        F.col("r.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("r.customer_id"),
        F.col("c.city"),
        F.col("c.loyalty_tier"),
        F.col("c.preferred_category"),
        F.col("r.recency_days"),
        F.col("r.frequency"),
        F.col("r.monetary_value"),
        F.col("r.avg_order_value"),
        F.col("r.stores_visited"),
        F.col("r.r_score"),
        F.col("r.f_score"),
        F.col("r.m_score"),
        F.col("r.rfm_score"),
        F.col("r.rfm_segment"),
        F.col("c.churn_risk"),
        F.col("c.customer_segment"),
        F.col("r.last_purchase_date"),
        F.col("r.gold_processed_at"),
    )
)

print(f"Final RFM rows: {df_rfm_final.count():,}")

In [0]:
# ── Write Gold + register ─────────────────────────────────────
(
    df_rfm_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("rfm_segment")
    .save(f"{GOLD_PATH}gold_customer_rfm/")
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_gold.gold_customer_rfm
    USING DELTA
    LOCATION '{GOLD_PATH}gold_customer_rfm/'
""")

print("✅ walmart_gold.gold_customer_rfm registered")
print(f"   Rows: {spark.table('walmart_gold.gold_customer_rfm').count():,}")

# Final business summary
spark.sql("""
    SELECT
        rfm_segment,
        COUNT(*)                        AS customers,
        ROUND(SUM(monetary_value), 2)   AS total_revenue,
        ROUND(AVG(monetary_value), 2)   AS avg_revenue_per_customer,
        ROUND(AVG(frequency), 1)        AS avg_orders
    FROM walmart_gold.gold_customer_rfm
    GROUP BY rfm_segment
    ORDER BY total_revenue DESC
""").display()